In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from pyspark.sql import Row

# 1. Configurações Iniciais
meu_catalogo = "workspace"
caminho_volume = "/Volumes/workspace/default/rocket1"

# 2. Criar o banco de dados (esquema) da camada Bronze 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {meu_catalogo}.bronze")

# 3. Mapeamento de Arquivos para Tabelas Bronze [cite: 17, 20]
mapeamento_tabelas = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

# 4. Ingestão dos CSVs com timestamp_ingestion [cite: 14, 15]
for arquivo, nome_tabela in mapeamento_tabelas.items():
    try:
        path = f"{caminho_volume}/{arquivo}"
        # Lendo o CSV
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(path)
        
        # Adicionando o timestamp do momento da inserção [cite: 15]
        df = df.withColumn("timestamp_ingestion", current_timestamp())
        
        # Salvando na camada bronze
        df.write.format("delta").mode("overwrite").saveAsTable(f"{meu_catalogo}.bronze.{nome_tabela}")
        print(f"✅ Tabela {nome_tabela} carregada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao carregar {arquivo}: {e}")

# 5. Ingestão de API: Cotação do Dólar [cite: 21]
# Configuração de parâmetros para datas 
dbutils.widgets.text("data_inicio", "01-01-2016")
dbutils.widgets.text("data_fim", "12-31-2018")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# Endpoint do Banco Central [cite: 23]
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

try:
    response = requests.get(url)
    data_json = response.json().get('value', [])
    
    if data_json:
        df_api = spark.createDataFrame([Row(**item) for item in data_json])
        # Adicionando timestamp_ingestion para a API também
        df_api = df_api.withColumn("timestamp_ingestion", current_timestamp())
        
        df_api.write.format("delta").mode("overwrite").saveAsTable(f"{meu_catalogo}.bronze.tb_cotacao_dolar")
        print("✅ Ingestão da API de cotação concluída!")
    else:
        print("⚠️ API não retornou dados para o período selecionado.")
except Exception as e:
    print(f"❌ Erro na ingestão da API: {e}")

✅ Tabela tb_customers carregada com sucesso!
✅ Tabela tb_geolocalizacao carregada com sucesso!
✅ Tabela tb_order_items carregada com sucesso!
✅ Tabela tb_order_payments carregada com sucesso!
✅ Tabela tb_order_reviews carregada com sucesso!
✅ Tabela tb_orders carregada com sucesso!
✅ Tabela tb_products carregada com sucesso!
✅ Tabela tb_sellers carregada com sucesso!
✅ Tabela tb_product_category_name_translation carregada com sucesso!
✅ Ingestão da API de cotação concluída!
